In [31]:
from pathlib import Path
from reducto import Reducto
import os
from dotenv import load_dotenv

load_dotenv()  # Load env vars from .env

my_key = os.getenv("REDUCTO_API_KEY")

client = Reducto(api_key=my_key)
upload = client.upload(file=Path("sample.pdf"))

result = client.parse.run(
  document_url=upload,
  options={ # Sample configurations
    "ocr_mode": "agentic",
    "extraction_mode": "ocr",
    "chunking": {
        "chunk_mode": "variable",
    }
  },
  advanced_options={
    "ocr_system": "multilingual",
    "page_range": {
        "start": 1,
        "end": 10,
    },
    "table_output_format": "ai_json",
    "merge_tables": True,
  },
  experimental_options={
    "enable_checkboxes": True,
    "return_figure_images": False,
    "rotate_pages": True,
  }
)

print(result)

ParseResponse(duration=45.02127385139465, job_id='5199b8dd-e2b7-4cb6-9d52-ac8e4f239b51', result=ResultFullResult(chunks=[ResultFullResultChunk(blocks=[ResultFullResultChunkBlock(bbox=BoundingBox(height=0.9444444444444444, left=0.008986928104575163, page=1, top=0.0422979797979798, width=0.9722222222222222, original_page=1), content='[["", "Trial System - To print your firm name here, call Thomson Reuters at 800-968-8900.,"]]', type='Table', confidence='high', image_url=None), ResultFullResultChunkBlock(bbox=BoundingBox(height=0.018308080808080808, left=0.41748366013071897, page=2, top=0.07828282828282829, width=0.18055555555555555, original_page=2), content='Filing Instructions', type='Title', confidence='high', image_url=None), ResultFullResultChunkBlock(bbox=BoundingBox(height=0.011994949494949494, left=0.41748366013071897, page=2, top=0.1167929292929293, width=0.18137254901960784, original_page=2), content='FinCEN Form 114', type='Title', confidence='high', image_url=None)], content=

In [32]:
parsed_data = result.model_dump()


In [33]:
with open("parsed_data.txt", "w", encoding="utf-8") as f:
    f.write(str(parsed_data))

In [67]:
# parsed_data create a txt file out of this variable
import json

with open("parsed_data.txt") as f:
    data = f.read()

# If your file is not valid JSON, you may need to preprocess it.
# Assuming it's a Python dict as a string:
parsed = eval(data)  # Use json.loads(data) if it's valid JSON

page3_blocks = []
for chunk in parsed['result']['chunks']:
    for block in chunk.get('blocks', []):
        if block.get('bbox', {}).get('page') == 6:
            page3_blocks.append(block)
printed_text = ""
# Print or process page 3 blocks
for block in page3_blocks:
    printed_text = printed_text+block['content']



In [68]:
printed_text

'A2LGENA 07/29/2025 3:33 PMForm 1040FinCEN 114 - Report of Foreign Bank and Financial Accounts, Page 12024NameFname GenInfoTaxpayer Identification Number: 123-65-4987Warning: Printed versions of the BSA E-Filing forms are not for submission and will not be processed by FinCENSpaces and dashes have been removed from identification numbers and postal codes where required for FinCEN electronic filing.1 This report is for calendar year ended 12/31/: 2024\nAmended: [ ]\nPrior report BSA Identifier: <empty>\nReason if filing late: <empty>Part I - Filer Information[["2", "Type of filer", "Individual"], ["3", "U.S. Taxpayer Identification Number", "123654987"], ["3a", "TIN type", "SSN/ITIN"], ["4", "Foreign identification", ""], ["", "4a Type Passport", ""], ["", "4b Number", "24131411"], ["", "4c Country of Issue", "Falkland Islands (Isla"], ["5", "Individual\'s date of birth", "06/06/1979"], ["6", "Last name or organization", "GenInfo"], ["name\\n7 First name Fname", "", ""], ["8", "Middle i

In [64]:
s = page3_blocks[10]['content']
# s = page3_blocks[11]

# dict_block = page3_blocks[11]['content'].to_dict()  # Convert to dict if needed

In [75]:
import re
from typing import Any, Dict, List

EMPTY_TOKENS = {"", "<empty>", "\\u2014", "\u2014", "—", "-"}

def _clean(s: Any) -> str:
    if s is None: return ""
    s = str(s).strip()
    return "" if s in EMPTY_TOKENS else s

def _is_placeholder(s: Any) -> bool:
    if s is None: return True
    return str(s).strip() in EMPTY_TOKENS

def _after_colon(line: str) -> str:
    m = re.search(r":\s*(.*)$", line)
    return _clean(m.group(1)) if m else ""

def _calendar_ended(line: str) -> str:
    m = re.search(r"calendar year ended\s+(\d{1,2}/\d{1,2})/?\s*[:\-]?\s*(\d{4})", line, re.I)
    if m: return f"{m.group(1)}/{m.group(2)}"
    m1 = re.search(r"(\d{1,2}/\d{1,2})", line)
    m2 = re.search(r"(\d{4})(?!.*\d)", line)
    return f"{m1.group(1)}/{m2.group(1)}" if (m1 and m2) else ""

def _parse_table_blob(blob: str) -> Dict[str, Any]:
    out, foreign = {}, {}

    def grab(pat, g=1):
        m = re.search(pat, blob, re.I)
        return _clean(m.group(g)) if m else ""

    out["Type_of_filer"] = grab(r'"Type of filer"\s*,\s*"([^"]*)"')
    out["TIN"]            = grab(r'"U\.S\. Taxpayer Identification Number"\s*,\s*"([^"]*)"')
    out["TIN_TYPE"]       = grab(r'"TIN type"\s*,\s*"([^"]*)"')

    ftype = grab(r'"4a Type\s+([^"]+)"') or grab(r'"4a Type\s*([^"]+)"')
    if ftype: foreign["Type"] = ftype
    fnum  = grab(r'"4b Number"\s*,\s*"([^"]*)"')
    if fnum: foreign["Number"] = fnum
    fctry = grab(r'"4c Country of Issue"\s*,\s*"([^"]*)"')
    if fctry: foreign["Country_of_issue"] = fctry
    if foreign: out["Foreign_identification"] = foreign

    out["DOB"]                       = grab(r"\"Individual's date of birth\"\s*,\s*\"([^\".]*)\"")
    out["Last_name_or_organization"] = grab(r'"Last name or organization"\s*,\s*"([^"]*)"')

    m_fn = re.search(r'First name\s+([^\s\]",]+)', blob, re.I)
    out["First_Name"] = _clean(m_fn.group(1)) if m_fn else ""

    out["Middle_Initial"]  = _clean(grab(r'"Middle initial"\s*,\s*"([^"]*)"'))
    out["Suffix"]          = _clean(grab(r'"Suffix"\s*,\s*"([^"]*)"'))
    out["Mailing_address"] = grab(r'"Mailing address"\s*,\s*"([^"]*)"')
    out["City"]            = grab(r'"City"\s*,\s*"([^"]*)"')
    out["State"]           = grab(r'"State"\s*,\s*"([^"]*)"')
    out["zip_postal_code"] = grab(r'"Zip/postal code"\s*,\s*"([^"]*)"')

    mc = re.search(r'"Country\s+([A-Za-z]{2,})"', blob, re.I)
    out["Country"] = _clean(mc.group(1)) if mc else ""

    # 14a between 14a..14b
    seg_14a = ""
    m14a = re.search(r'(14a[^]]*\][^\[]*)(14b[^]]*)', blob, re.I)
    if m14a: seg_14a = m14a.group(1)
    else:
        m14a_only = re.search(r'(14a[^]]*\][\s\S]*)$', blob, re.I)
        seg_14a = m14a_only.group(1) if m14a_only else ""
    if seg_14a:
        yes_checked = bool(re.search(r'\["Yes",\s*"\s*\[(?:x|X)\]\s*"', seg_14a))
        no_checked  = bool(re.search(r'\["No",\s*"\s*\[(?:x|X)\]\s*"', seg_14a))
        out["FINANCIAL_Interest_accounts"] = True if yes_checked else False
        if no_checked: out["FINANCIAL_Interest_accounts"] = False
    else:
        out["FINANCIAL_Interest_accounts"] = False

    # 14b after 14b
    seg_14b_m = re.search(r'(14b[\s\S]*)$', blob, re.I)
    seg_14b = seg_14b_m.group(1) if seg_14b_m else ""
    if seg_14b:
        yes_b = bool(re.search(r'Yes[^"\]]*["\]]\s*[\[☑✓xX]', seg_14b))
        no_b  = bool(re.search(r'No[^"\]]*[\u2611☑✓xX]', seg_14b))
        out["Sign_Auth_no_interest"] = False if no_b else True if yes_b else False
    else:
        out["Sign_Auth_no_interest"] = False

    # normalize placeholders to empty
    for k in ("Middle_Initial", "Suffix"):
        if _is_placeholder(out.get(k)): out[k] = ""
    return out

def parse_page3_blocks_resilient(page3_blocks: List[Dict[str, Any]]) -> Dict[str, Any]:
    result: Dict[str, Any] = {}
    expect_name = False
    accumulating = False
    table_buf: List[str] = []

    for blk in page3_blocks:
        content = blk.get("content")
        if not isinstance(content, str):
            # if you sometimes get an actual list-of-rows instead of a string blob, handle here if needed
            continue

        raw = content
        line = raw.strip()
        if not line: continue

        # already accumulating table?
        if accumulating:
            table_buf.append(raw)
            joined = "\n".join(table_buf)
            if joined.count("[") - joined.count("]") <= 0:
                result["Filer_Information"] = _parse_table_blob(joined)
                accumulating = False
            continue

        # split if header is glued
        if "Part I - Filer Information" in line:
            before, after = line.split("Part I - Filer Information", 1)
            before = before.strip()
            if before:
                if expect_name:
                    result["Name"] = _clean(before); expect_name = False
                elif before.lower().startswith("taxpayer identification number"):
                    result["Taxpayer Identification Number"] = _after_colon(before)
                elif "calendar year ended" in before.lower():
                    result["calendar_year_ended"] = _calendar_ended(before)
                elif before.lower().startswith("amended"):
                    result["Amended"] = bool(re.search(r"\[(x|X|☑|✓)\]", before))
                elif before.lower().startswith("prior report bsa identifier"):
                    v = _after_colon(before); result["BSA_identifier"] = "" if _is_placeholder(v) else v
                elif before.lower().startswith("reason if filing late"):
                    v = _after_colon(before); result["Reason_if_filing_late"] = "" if _is_placeholder(v) else v
                elif re.match(r"^Form\s+(\d+)\b", before, re.I):
                    result["Form"] = re.match(r"^Form\s+(\d+)\b", before, re.I).group(1)
                elif re.fullmatch(r"\d{4}", before):
                    result["year"] = before
                elif before.lower() == "name":
                    expect_name = True

            trailing = after.strip()
            if trailing.startswith("[["):
                accumulating = True
                table_buf = [trailing]
                if trailing.count("[") - trailing.count("]") <= 0:
                    result["Filer_Information"] = _parse_table_blob(trailing)
                    accumulating = False
            else:
                accumulating = True
                table_buf = []
            continue

        # normal lines
        if expect_name:
            result["Name"] = _clean(line); expect_name = False; continue
        m = re.match(r"^Form\s+(\d+)\b", line, re.I)
        if m: result["Form"] = m.group(1); continue
        if re.fullmatch(r"\d{4}", line): result["year"] = line; continue
        if line.lower() == "name": expect_name = True; continue
        if line.lower().startswith("taxpayer identification number"):
            result["Taxpayer Identification Number"] = _after_colon(line); continue
        if "calendar year ended" in line.lower():
            result["calendar_year_ended"] = _calendar_ended(line); continue
        if line.lower().startswith("amended"):
            result["Amended"] = bool(re.search(r"\[(x|X|☑|✓)\]", line)); continue
        if line.lower().startswith("prior report bsa identifier"):
            v = _after_colon(line); result["BSA_identifier"] = "" if _is_placeholder(v) else v; continue
        if line.lower().startswith("reason if filing late"):
            v = _after_colon(line); result["Reason_if_filing_late"] = "" if _is_placeholder(v) else v; continue
        if line.startswith("[["):
            accumulating = True; table_buf = [raw]
            if raw.count("[") - raw.count("]") <= 0:
                result["Filer_Information"] = _parse_table_blob(raw); accumulating = False
            continue

    # defaults if truly missing
    result.setdefault("Amended", False)
    result.setdefault("BSA_identifier", "")
    result.setdefault("Reason_if_filing_late", "")

    # normalize em-dash placeholders in nested fields
    fi = result.get("Filer_Information", {})
    for k in ("Middle_Initial", "Suffix"):
        if _is_placeholder(fi.get(k)): fi[k] = ""

    return result


In [76]:
final = parse_page3_blocks_resilient(page3_blocks)

In [77]:
final

{'Form': '1040',
 'year': '2024',
 'Name': 'Fname GenInfo',
 'Taxpayer Identification Number': '123-65-4987',
 'calendar_year_ended': '12/31/2024',
 'Filer_Information': {'Type_of_filer': 'Individual',
  'TIN': '123654987',
  'TIN_TYPE': 'SSN/ITIN',
  'Foreign_identification': {'Type': 'Passport',
   'Number': '24131411',
   'Country_of_issue': 'Falkland Islands (Isla'},
  'DOB': '06/06/1979',
  'Last_name_or_organization': 'GenInfo',
  'First_Name': 'Fname',
  'Middle_Initial': '',
  'Suffix': '',
  'Mailing_address': 'apt24',
  'City': 'Iron Mountain',
  'State': 'MI Michigan',
  'zip_postal_code': '42929',
  'Country': 'US',
  'FINANCIAL_Interest_accounts': False,
  'Sign_Auth_no_interest': False},
 'Amended': False,
 'BSA_identifier': '',
 'Reason_if_filing_late': ''}